# Yield Estimation for Floating Rate Bonds

##### This notebook contains implementation of 5 methods for yield esimation for a floating-rate bond

1. Simple Margin (income from spread and bond discount received over the life of bond)
2. Last (calulcate unknown cashflows based on the last known rate)
3. Expert Forecast (calulcate unknown cashflows based on an expert forecast)
4. Forward (calulcate unknown cashflows based on forward rates from the swap market)
5. Adjusted Forward (same as #4 but with adjustments for interest rate capitalisation, averaging and observation lag)

More details on the methodology can be found in:

1. Makushkin (2027). Modelling the Yield of Floating Rate Bonds on the Russian Market
2. Makushkin (2025). Drivers of Yield of Russian Sovereign Bonds with Floating Coupons (OFZ-PK). Financial Journal, 17(5), 8–25 (In Russian).
https://ideas.repec.org/a/fru/finjrn/250501p8-25.html


In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from dateutil.relativedelta import relativedelta
from scipy import optimize
from scipy import interpolate
from matplotlib.ticker import PercentFormatter
from numpy.linalg import lstsq
%matplotlib inline
import matplotlib.dates as mdates

### Method 1. Simple Margin

In [2]:
def simple_margin(price, spread, maturity, freq):
    return ((1+(spread/price)/freq)**freq-1)+(((100/price))**(1/maturity)-1)

In [3]:
simple_margin(95, 5, 5, 1)

0.06294303826530445

### Method 2. Last Rate

In [4]:
def accrued_interest(maturity, coupon_rate, frequency):

    return (1/frequency - maturity % (1/frequency)) * coupon_rate * 100

def ytm_float_last(price, last_rate, maturity,
                 spread, frequency, face_value=100, guess=0.1):

    T = maturity
    periods = T // (1/frequency) + 1
    ost = T % (1/frequency)  # Remaining fractional period (if applicable)
    coupon = (last_rate+spread) * face_value / frequency  # Calculate coupon payment
    accrued = accrued_interest(maturity, last_rate+spread, frequency)

    # Define the YTM function with continuous compounding
    ytm_func = lambda y: np.sum(
            [coupon / (1 + y) ** (ost + 1/frequency * t) for t in range(0, int(round(periods)))]) + 100 / ((1 + y) ** T) - (price + accrued)

    # Use Newton-Raphson (scipy.optimize.newton) to find the continuous YTM
    ytm = optimize.newton(ytm_func, guess, disp=False)
    continuous_yield = np.log(1 + ytm)

    return ytm #continues yield is used instead of annually compunded to be in line with NS model specification

In [5]:
ytm_float_last(price=99,
               last_rate=0.1,
               maturity=5.5,
               spread=0.01,
               frequency=12)

0.11842636747272685

### Method 3. Expert Forecast

In [6]:
# Helper: Linear interpolation with flat extrapolation on the short end
def interpolate_zero_rate(maturities, zero_rates, t):
    if t <= maturities[0]:
        return zero_rates[0]
    elif t >= maturities[-1]:
        return zero_rates[-1]
    else:
        for i in range(1, len(maturities)):
            if t < maturities[i]:
                t1, r1 = maturities[i - 1], zero_rates[i - 1]
                t2, r2 = maturities[i], zero_rates[i]
                return r1 + (r2 - r1) * (t - t1) / (t2 - t1)

In [7]:
def ytm_float_expert(price, maturities, zero_rates, maturity,
                  spread, frequency, face_value=100, guess=0.1):

    T = maturity
    dt = 1 / frequency
    ost = T % (dt)  # Remaining fractional period (if applicable)
    periods = int(maturity / dt) + 1
    times = [ost + dt * i for i in range(periods)]

    cash_flows = []
    for i, t2 in enumerate(times):
        t1 = times[i - 1] if i > 0 else 0
        z = interpolate_zero_rate(maturities, zero_rates, t2)
        coupon = (z+spread) * face_value * (t2-t1)
        cash_flows.append((t2, coupon))

    # Add principal to the last cash flow
    cash_flows[-1] = (cash_flows[-1][0], cash_flows[-1][1] + face_value)

    # Define YTM function
    ytm_func = lambda y: sum(cf / (1 + y) ** t for t, cf in cash_flows) - price

    # Solve using Newton-Raphson
    ytm = optimize.newton(ytm_func, guess, disp=False)
    return ytm

In [8]:
ytm_float_expert(price=99,
                 maturities=(0.1, 0.5, 1, 3, 5, 10), 
                 zero_rates=(0.1, 0.11, 0.12, 0.13, 0.14, 0.15),
                 maturity=5, 
                 spread=0.01,
                 frequency=12)

0.14608103233250505

### Method 4. Forward rates

In [9]:
# Helper: Forward rate from zero rates (continuous compounding)
def forward_rate(z1, t1, z2, t2):
    if abs(t2 - t1) < 1e-8:
        return z2  # fallback to flat rate or z2 as approximation
    return (z2 * t2 - z1 * t1) / (t2 - t1)

In [10]:
def ytm_float_fwd(price, maturities, zero_rates, maturity,
                  spread, frequency, face_value=100, guess=0.1):

    #averaging = 1 if TRUE or 0 if FALSE
    #lag expressed in days
    
    T = maturity
    dt = 1 / frequency
    ost = T % (dt)  # Remaining fractional period (if applicable)
    periods = int(maturity / dt) + 1
    times = [ost + dt * i for i in range(periods)]

    # Build floating cash flows using forward rates
    cash_flows = []
    for i, t2 in enumerate(times):
        t1 = times[i - 1] if i > 0 else 0
        z1 = interpolate_zero_rate(maturities, zero_rates, t1)
        z2 = interpolate_zero_rate(maturities, zero_rates, t2)
        fwd = forward_rate(z1, t1, z2, t2)
            
        coupon = (fwd+spread) * face_value * (t2-t1)
        cash_flows.append((t2, coupon))

    # Add principal to the last cash flow
    cash_flows[-1] = (cash_flows[-1][0], cash_flows[-1][1] + face_value)

    # Define YTM function
    ytm_func = lambda y: sum(cf / (1 + y) ** t for t, cf in cash_flows) - price

    # Solve using Newton-Raphson
    ytm = optimize.newton(ytm_func, guess, disp=False)
    return ytm

In [11]:
ytm_float_fwd(price=99,
                 maturities=(0.1, 0.5, 1, 3, 5, 10), 
                 zero_rates=(0.1, 0.11, 0.12, 0.13, 0.14, 0.15),
                 maturity=5, 
                 spread=0.01,
                 frequency=12)

0.16038946304931645

### Method 5. Volatility Adjustments

In [12]:
def ytm_float_vol(price, maturities, zero_rates, maturity,
                  vol, spread, frequency, averaging, lag, face_value=100, guess=0.1):

    #averaging = 1 if TRUE or 0 if FALSE
    #lag expressed in days
    
    T = maturity
    dt = 1 / frequency
    ost = T % (dt)  # Remaining fractional period (if applicable)
    periods = int(maturity / dt) + 1
    times = [ost + dt * i for i in range(periods)]

    # Build floating cash flows using forward rates
    cash_flows = []
    for i, t2 in enumerate(times):
        t1 = times[i - 1] if i > 0 else 0
        z1 = interpolate_zero_rate(maturities, zero_rates, t1)
        z2 = interpolate_zero_rate(maturities, zero_rates, t2)
        fwd = forward_rate(z1, t1, z2, t2)
        
        cap_adj = (1/frequency)/2*(fwd**2)
        convex_adj =(1/frequency)/2 * (vol**2)*t2
        lag_adj_avg = (fwd**2)*(vol**2)*(1/(2*frequency)+lag/365)*t2/(1+fwd*(1/(2*frequency)+lag/365))
        lag_adj_simple = (fwd**2)*(vol**2)*(lag/365)*t2/(1+fwd*(lag/365))
        
        if averaging == 1:
            adj = cap_adj+convex_adj+lag_adj_avg
        else:
            adj = lag_adj_simple
            
        coupon = (fwd+spread-adj) * face_value * (t2-t1)
        cash_flows.append((t2, coupon))

    # Add principal to the last cash flow
    cash_flows[-1] = (cash_flows[-1][0], cash_flows[-1][1] + face_value)

    # Define YTM function
    ytm_func = lambda y: sum(cf / (1 + y) ** t for t, cf in cash_flows) - price

    # Solve using Newton-Raphson
    ytm = optimize.newton(ytm_func, guess, disp=False)
    return ytm

In [13]:
ytm_float_vol(price=99,
              maturities=(1,5,10), 
              zero_rates=(0.1, 0.1, 0.1),
              maturity=5, 
              vol=0.05, 
              spread=0.01,
              frequency=12,
              averaging=1,
              lag=5)

0.1178880286377021